## 1. Import required Libraries

In [14]:
!pip install xgboost scikit-learn pandas cleanlab
!pip install importnb
!pip install tensorflow
!pip install opencv-contrib-python

In [15]:
import sys
import os
sys.path.append(os.path.abspath("C:/Users/Admin/OneDrive - University of Huddersfield/Data-driven-AI/Assignment 2"))

In [16]:
import tensorflow as tf
import cv2
import cleanlab
import shutil

## 2. Loading the Data

In [18]:

# ==============================================
# 1. PATH SETUP
# ==============================================

repo_path = 'C:/Users/Admin/OneDrive - University of Huddersfield/Data-driven-AI/Assignment 2/CHS2406_Coursework2_Data_Repository'
dataset_path = 'C:/Users/Admin/OneDrive - University of Huddersfield/Data-driven-AI/Assignment 2/dataset'

processed_image_count = 0
count_label_loading_error = 0

os.makedirs(dataset_path, exist_ok=True)

output_file = 'image_labels.txt'

labels = []
corrupted_images = []
unlabeled_images = []

print("===== COPYING IMAGES & ASSIGNING LABELS =====\n")

# ==============================================
# 2. MAIN LOOP: COPY + LABEL IN ONE GO
# ==============================================

label_map = {}  # StageN → N

for folder_name in sorted(os.listdir(repo_path)):
    source_folder = os.path.join(repo_path, folder_name)

    if not os.path.isdir(source_folder):
        continue  # skip file tại root

    # ----------------------------------------
    # Xác định folder dạng StageN (stage1, stage2, ...)
    # ----------------------------------------
    if folder_name.lower().startswith("stage"):
        try:
            stage_number = int(folder_name[5:])
            label_map[folder_name] = stage_number
        except:
            print(f"⚠️ Warning: Folder {folder_name} is not valid StageN.")
            continue

        current_label_int = label_map[folder_name]

        # ----------------------------------------
        # Duyệt ảnh và xử lý ngay
        # ----------------------------------------
        for filename in os.listdir(source_folder):

            source_file = os.path.join(source_folder, filename)

            if not filename.lower().endswith(('.jpg', '.jpeg', '.png')):
                # Không dùng source_file trước khi khai báo!
                print(f"⚠️ Skipping non-image file: {source_file}")
                count_label_loading_error += 1
                continue

            img = cv2.imread(source_file)

            if img is None:
                corrupted_images.append(os.path.join(folder_name, filename))
                print(f"❌ Cannot read (corrupted): {source_file}")
                count_label_loading_error += 1
                continue

            # Copy ảnh hợp lệ vào dataset
            dst = os.path.join(dataset_path, filename)
            shutil.copy2(source_file, dst)
            processed_image_count += 1

            # Gán label ngay lập tức
            labels.append((filename, current_label_int-1))

    # ----------------------------------------
    # Folder không phải StageN → ảnh không gán nhãn
    # ----------------------------------------
    else:
        for filename in os.listdir(source_folder):
            if filename.lower().endswith(('.jpg', '.jpeg', '.png')):
                unlabeled_images.append(os.path.join(folder_name, filename))


print("\nTotal successfully processed images: ", processed_image_count)
print("Number of unreadable images: ", count_label_loading_error)
print("Total images scanned: ", processed_image_count + count_label_loading_error)
print("\n===== FINISHED COPYING & LABELING =====\n")

# ==============================================
# 3. SAVE LABEL FILE
# ==============================================

with open(output_file, "w", encoding="utf-8") as f:
    for filename, label in labels:
        f.write(f"{filename}: {label}\n")

print(f"🎯 Saved {len(labels)} labeled images to {output_file}\n")


# ==============================================
# 4. REPORT
# ==============================================

if corrupted_images:
    print("❌ Corrupted images:")
    for x in corrupted_images:
        print(" -", x)

if unlabeled_images:
    print("\n⚠️ Images in non-Stage folders (not labeled):")
    for x in unlabeled_images:
        print(" -", x)

if not corrupted_images and not unlabeled_images:
    print("🎉 All images were valid and assigned labels correctly!")

===== COPYING IMAGES & ASSIGNING LABELS =====

⚠️ Skipping non-image file: C:/Users/Admin/OneDrive - University of Huddersfield/Data-driven-AI/Assignment 2/CHS2406_Coursework2_Data_Repository\Stage1\stage 1 _ 7 u2362759.heic
⚠️ Skipping non-image file: C:/Users/Admin/OneDrive - University of Huddersfield/Data-driven-AI/Assignment 2/CHS2406_Coursework2_Data_Repository\Stage1\stage 1_ 10 u2362759.heic
⚠️ Skipping non-image file: C:/Users/Admin/OneDrive - University of Huddersfield/Data-driven-AI/Assignment 2/CHS2406_Coursework2_Data_Repository\Stage1\stage 1_1 u2362759.heic
⚠️ Skipping non-image file: C:/Users/Admin/OneDrive - University of Huddersfield/Data-driven-AI/Assignment 2/CHS2406_Coursework2_Data_Repository\Stage1\stage 1_2 u2362759.heic
⚠️ Skipping non-image file: C:/Users/Admin/OneDrive - University of Huddersfield/Data-driven-AI/Assignment 2/CHS2406_Coursework2_Data_Repository\Stage1\stage 1_3 u2362759.heic
⚠️ Skipping non-image file: C:/Users/Admin/OneDrive - University of H

## 3. Data Labelling Errors

In [ ]:
# ============================================================
# FIX: TẢI LẠI DỮ LIỆU NẾU KHÔNG TÌM THẤY TRONG BỘ NHỚ
# ============================================================

# Giả định dataset_path, output_file và labels được định nghĩa ở cell trước.
# Nếu không, ta cần tải lại chúng.
if 'labels' not in globals() or 'dataset_path' not in globals() or not labels:
    print("⚠️ Cảnh báo: Biến 'labels' không tồn tại trong bộ nhớ. Đang thử tải từ file...")
    
    # Cần khai báo lại các biến đường dẫn nếu kernel bị reset
    try:
        if 'dataset_path' not in globals():
            # Giả định lại đường dẫn từ code gốc của bạn
            dataset_path = 'C:/Users/Admin/OneDrive - University of Huddersfield/Data-driven-AI/Assignment 2/dataset'
        if 'output_file' not in globals():
            output_file = 'image_labels.txt'
            
        labels = []
        with open(output_file, "r", encoding="utf-8") as f:
            for line in f:
                # Định dạng file: filename: label_int
                parts = line.strip().split(': ')
                if len(parts) == 2:
                    filename = parts[0]
                    label = int(parts[1]) # Nhãn 0-7
                    labels.append((filename, label))
        
        if not labels:
             print("❌ LỖI: File 'image_labels.txt' trống hoặc không đọc được dữ liệu nhãn.")
             sys.exit()
             
        print(f"✅ Tải thành công {len(labels)} nhãn từ file '{output_file}'.")
        
    except FileNotFoundError:
        print(f"❌ LỖI: Không tìm thấy file nhãn '{output_file}'. Vui lòng chạy lại các cell 1 và 2.")
        sys.exit()
    except Exception as e:
        print(f"❌ LỖI khi tải nhãn: {e}")
        sys.exit()
# ============================================================

In [ ]:
# ============================================================
# 3. DATA LABELLING ERRORS DETECTION WITH CLEANLAB
# ============================================================

import numpy as np
import cv2
import pandas as pd
import os
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import cross_val_predict
from cleanlab.filter import find_label_issues
# Thư viện cho Feature Extraction (Transfer Learning)
from tensorflow.keras.applications.mobilenet_v2 import MobileNetV2, preprocess_input
from tqdm.notebook import tqdm
import sys

print("=== Bắt đầu phát hiện lỗi gán nhãn với Cleanlab ===")

# 1. Chuẩn bị dữ liệu (labels đã là 0-7)
file_names = [item[0] for item in labels]
# Nhãn 0-7
y_labels = np.array([item[1] for item in labels]) 
full_paths = [os.path.join(dataset_path, fname) for fname in file_names]

# 2. Hàm load và tiền xử lý ảnh (Resize về 224x224 cho MobileNetV2)
def load_and_preprocess_images(image_paths, target_size=(224, 224)):
    """Loads, resizes images, and handles loading errors."""
    images = []
    print("📸 Loading images and resizing (224x224) for feature extraction...")
    valid_indices = []
    for idx, path in enumerate(tqdm(image_paths)):
        img = cv2.imread(path)
        if img is not None:
            img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB) 
            img = cv2.resize(img, target_size)
            images.append(img)
            valid_indices.append(idx)
        
    return np.array(images), valid_indices

# Load ảnh và lấy chỉ mục ảnh hợp lệ
X_images, valid_indices = load_and_preprocess_images(full_paths)

In [7]:
print("✨ Extracting features using MobileNetV2...")

base_model = MobileNetV2(weights='imagenet', include_top=False, pooling='avg')
X_images_preprocessed = preprocess_input(X_images)
X_features = base_model.predict(X_images_preprocessed, verbose=1)

print("✨ Training simple baseline model (Logistic Regression)...")
model = LogisticRegression(max_iter=2000)

pred_probs = cross_val_predict(
    model, 
    X_features, 
    y_labels[valid_indices], 
    cv=5,
    method='predict_proba'
)

label_issues = find_label_issues(
    labels=y_labels[valid_indices],
    pred_probs=pred_probs
)

print("🔍 Cleanlab flagged:", label_issues.sum(), "potential label errors")


✨ Extracting features using MobileNetV2...


C:\Users\Admin\AppData\Local\Temp\ipykernel_6620\3594389685.py:3: UserWarning: `input_shape` is undefined or non-square, or `rows` is not in [96, 128, 160, 192, 224]. Weights for input shape (224, 224) will be loaded as the default.
  base_model = MobileNetV2(weights='imagenet', include_top=False, pooling='avg')


267/267 ━━━━━━━━━━━━━━━━━━━━ 139s 504ms/step
✨ Training simple baseline model (Logistic Regression)...
🔍 Cleanlab flagged: 4945 potential label errors


## Label errors:
**Explain what kind of errors you found in the dataset.**
1. Incorrect data size
2. Wrong file type

**List the total number of images left in each class/stage after the label error handling**

<br>

<ol>
  <li>Stage 1: <<Number of images>></li>
  <li>Stage 2: <<Number of images>></li>
  <li>Stage 3: <<Number of images>></li>
  <li>Stage 4: <<Number of images>></li>
  <li>Stage 5: <<Number of images>></li>
  <li>Stage 6: <<Number of images>></li>
  <li>Stage 7: <<Number of images>></li>
  <li>Stage 8: <<Number of images>></li>
</ol>

## 4. Pre-process the Dataset

In [8]:
# <<insert yout code here>>

## 5. Split the data
<br>

Split the data into training, validation and testing dataset using Startification, ensuring equal class distribution.

Choose appropriate values of training, validation and testing datasets.

Display total number of images in each dataset split.

In [9]:
# <<insert yout code here>>

## 6. Model Implementation

In [10]:
# <<insert yout code here>>

## 7. Evaluate the Model

In [11]:
# <<insert yout code here>>

### Training Curves

In [12]:
# <<insert yout code here>>

### Make Inference
For some unseen data, make predictions using the trained model.

In [13]:
# <<insert yout code here>>